In [1]:
!pip install ngsolve
!pip install CoolProp


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 35.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 34.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 42.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 96.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ngsolve]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 23.7 MB/s  0:00:00


In [12]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.geom2d import unit_square
from math import pi, log
import CoolProp
from CoolProp.CoolProp import PropsSI
import numpy as np



In [13]:
def newton(U, a, f, freedofs, tol=1e-10, maxits=20, verbose=False):
    """
    solve a(U; v) = f(v)
    modifica U inplace
    """
    res = U.vec.CreateVector()
    du  = U.vec.CreateVector()
    for it in range(maxits):
        a.Apply(U.vec, res)                  # res_i = a(U; phi_i)
        res.data -= f.vec                    # R(U) = a(U; .) - f
        a.AssembleLinearization(U.vec)       # a.mat = dR/dU evaluado en U
        du.data = a.mat.Inverse(freedofs, inverse="umfpack") * res
        U.vec.data -= du                     # du = 0 en los dofs no libres
        err = Norm(du) / max(Norm(U.vec), 1.0)
        if verbose:
            print(f"  it {it:2d}: |du|/|U| = {err:.2e}")
        if err < tol:
            return it + 1
    raise RuntimeError(f"Newton no convergió en {maxits} iteraciones (|du|/|U| = {err:.2e})")


from netgen.geom2d import SplineGeometry

def malla_referencia(radio, altura, h0, maxh):
    geo = SplineGeometry()
    p = [geo.AppendPoint(*q) for q in [(0, h0), (radio, h0), (radio, altura), (0, altura)]]
    geo.Append(["line", p[0], p[1]], bc="liquido", leftdomain=1, rightdomain=0)
    geo.Append(["line", p[1], p[2]], bc="pared",   leftdomain=1, rightdomain=0)
    geo.Append(["line", p[2], p[3]], bc="techo",   leftdomain=1, rightdomain=0)
    geo.Append(["line", p[3], p[0]], bc="eje",     leftdomain=1, rightdomain=0)
    return Mesh(geo.GenerateMesh(maxh=maxh))


def prop_sat(prop, Q):
    return PropsSI(prop, "P", P_atm, "Q", Q, fluido)

mesh = malla_referencia(1.0, 2.0, 1.4, maxh=0.06)
print(mesh.GetBoundaries())


('liquido', 'pared', 'techo', 'eje')


In [14]:
# amoníaco a P = 1 atm <- CoolProp

fluido = "Ammonia"
P_atm  = 101325.0                              # [Pa]

def prop_sat(prop, Q):
    return PropsSI(prop, "P", P_atm, "Q", Q, fluido)

T_sat  = prop_sat("T", 0)                      # [K] interfase líquido-vapor
rho_v  = prop_sat("D", 1)                      # [kg/m^3]
cp_v   = prop_sat("C", 1)                      # [J/(kg K)]
k_v    = prop_sat("L", 1)                      # [W/(m K)]
rho_cp = rho_v * cp_v                          # [J/(m^3 K)] capacidad calórica volumétrica
rho_l  = prop_sat("D", 0)                      # [kg/m^3]  (etapa 9)
h_lv   = prop_sat("H", 1) - prop_sat("H", 0)   # [J/kg] calor latente (etapa 9)

# Rad con modelo P1
sigma = 5.670374419e-8                         # [W/(m^2 K^4)] Stefan-Boltzmann
kappa = 0.05                                   # [1/m] coef. de absorción (por estimar mejor)
D     = 1.0 / (3.0 * kappa)                    # [m] difusividad radiativa P1

# geo malla de ref
radio, altura, h0 = 1.0, 2.0, 1.4              # [m] h0: nivel inicial del líquido

# ---------- Datos por borde ----------
# T_pared = 20.0 + 273.15                      # [K]
T_pared = 5 + T_sat                            # [K]
T_techo = T_pared
T_borde   = {"liquido": T_sat, "pared": T_pared, "techo": T_techo}   # T de cada superficie (Marshak)
eps_borde = {"liquido": 0.95,                  #TODO: por estimar
             "pared":   0.8,
             "techo":   0.8}
gam_borde = {b: e / (2.0 * (2.0 - e)) for b, e in eps_borde.items()}  # coef. de Marshak

# ---------- Dirichlet de T y condición inicial compatible ----------
T_dirichlet = mesh.BoundaryCF({"liquido": T_sat, "techo": T_techo}, default=0)
T_inicial   = T_sat + (T_techo - T_sat) * (y - h0) / (altura - h0)

print(f"T_sat = {T_sat:.3f} K | rho_v = {rho_v:.4f} kg/m3 | cp_v = {cp_v:.1f} J/(kg K) | k_v = {k_v:.5f} W/(m K)")
print(f"rho_l = {rho_l:.2f} kg/m3 | h_lv = {h_lv/1e3:.1f} kJ/kg")


T_sat = 239.834 K | rho_v = 0.8900 kg/m3 | cp_v = 2291.6 J/(kg K) | k_v = 0.01953 W/(m K)
rho_l = 681.63 kg/m3 | h_lv = 1369.7 kJ/kg


In [15]:
# FEM
orden = 2
V   = H1(mesh, order=orden, dirichlet="liquido|techo")
W   = H1(mesh, order=orden)
# en U = (T, G)
fes = V * W
(T, G), (vT, vG) = fes.TnT()

U   = GridFunction(fes)
T_n = GridFunction(V) # T paso prev
dt  = Parameter(3600.0)

# a(U; v) "Bilineal"
# no lineal en U, lineal en (vT, vG)
T4 = T * T * T * T
a = BilinearForm(fes, symmetric=False)
a += (rho_cp / dt * T * vT
      + k_v * grad(T) * grad(vT)
      - kappa * (G - 4 * sigma * T4) * vT) * dx
a += (D * grad(G) * grad(vG)
      + kappa * G * vG
      - 4 * kappa * sigma * T4 * vG) * dx
# Marshak depende de G
for b, gam in gam_borde.items():
    a += gam * G * vG * ds(b)

# Forma lineal f(v)
f = LinearForm(fes)
f += rho_cp / dt * T_n * vT * dx
for b, gam in gam_borde.items():
    f += gam * 4 * sigma * T_borde[b]**4 * vG * ds(b)   # Marshak parte con T de la superficie


In [16]:
# Dirichlet en T
dofs_dirichlet_T = np.array([not V.FreeDofs()[i] for i in range(V.ndof)])
T_aux = GridFunction(V)
T_aux.Set(T_dirichlet, BND)

def aplicar_dirichlet(U):
    U.components[0].vec.FV().NumPy()[dofs_dirichlet_T] = T_aux.vec.FV().NumPy()[dofs_dirichlet_T]

# set T0
U.components[0].Set(T_inicial)
aplicar_dirichlet(U)
T_n.vec.data = U.components[0].vec
f.Assemble()

solo_G = BitArray(fes.ndof)
solo_G.Clear()
for i in range(fes.Range(1).start, fes.Range(1).stop):
    solo_G[i] = True

print("start G:")
newton(U, a, f, solo_G, verbose=True)


start G:
  it  0: |du|/|U| = 9.56e-01
  it  1: |du|/|U| = 5.22e-15


2

In [17]:
T_h, G_h = U.components
liq = mesh.Boundaries("liquido")
area_liq = Integrate(1, mesh, BND, definedon=liq)

q_cond = k_v * (T_techo - T_sat) / (altura - h0)
q_rad  = Integrate(gam_borde["liquido"] * (G_h - 4 * sigma * T_sat**4),
                   mesh, BND, definedon=liq) / area_liq

print(f"q_cond inicial = {q_cond:.3f} W/m2")
print(f"q_rad  inicial = {q_rad:.3f} W/m2")


q_cond inicial = 0.163 W/m2
q_rad  inicial = 15.487 W/m2


In [18]:
# dt.Set(3600.0)
dt.Set(60.0)
# n_pasos = 48
n_pasos = 60*4

# T_hist = GridFunction(V, multidim=0)
# T_hist.AddMultiDimComponent(U.components[0].vec)
# G_hist = GridFunction(V, multidim=0)
# G_hist.AddMultiDimComponent(U.components[1].vec)

T_hist = GridFunction(V, multidim=0)
G_hist = GridFunction(W, multidim=0)
T_hist.AddMultiDimComponent(T_h.vec)
G_hist.AddMultiDimComponent(G_h.vec)



for n in range(n_pasos):
    T_n.vec.data = U.components[0].vec     # update T_n
    f.Assemble()                           # f = f(T_n)
    aplicar_dirichlet(U)
    its = newton(U, a, f, fes.FreeDofs())
    # T_hist.AddMultiDimComponent(U.components[0].vec)
    # T_hist.AddMultiDimComponent(U.components[1].vec)
    T_hist.AddMultiDimComponent(T_h.vec)
    G_hist.AddMultiDimComponent(G_h.vec)
    if (n + 1) % 6 == 0:
        q_rad = Integrate(gam_borde["liquido"] * (G_h - 4 * sigma * T_sat**4),
                          mesh, BND, definedon=liq) / area_liq
        print(f"t = {(n+1)*dt.Get()/3600:5.1f} h | Newton: {its} it | q_rad = {q_rad:.3f} W/m2")


t =   0.1 h | Newton: 3 it | q_rad = 15.490 W/m2
t =   0.2 h | Newton: 3 it | q_rad = 15.493 W/m2
t =   0.3 h | Newton: 2 it | q_rad = 15.495 W/m2
t =   0.4 h | Newton: 2 it | q_rad = 15.497 W/m2
t =   0.5 h | Newton: 2 it | q_rad = 15.499 W/m2
t =   0.6 h | Newton: 2 it | q_rad = 15.500 W/m2
t =   0.7 h | Newton: 2 it | q_rad = 15.501 W/m2
t =   0.8 h | Newton: 2 it | q_rad = 15.502 W/m2
t =   0.9 h | Newton: 2 it | q_rad = 15.502 W/m2
t =   1.0 h | Newton: 2 it | q_rad = 15.503 W/m2
t =   1.1 h | Newton: 2 it | q_rad = 15.503 W/m2
t =   1.2 h | Newton: 2 it | q_rad = 15.504 W/m2
t =   1.3 h | Newton: 2 it | q_rad = 15.504 W/m2
t =   1.4 h | Newton: 2 it | q_rad = 15.504 W/m2
t =   1.5 h | Newton: 2 it | q_rad = 15.505 W/m2
t =   1.6 h | Newton: 2 it | q_rad = 15.505 W/m2
t =   1.7 h | Newton: 2 it | q_rad = 15.505 W/m2
t =   1.8 h | Newton: 2 it | q_rad = 15.505 W/m2
t =   1.9 h | Newton: 2 it | q_rad = 15.505 W/m2
t =   2.0 h | Newton: 2 it | q_rad = 15.505 W/m2
t =   2.1 h | Newton

In [19]:
Draw(T_hist, mesh, "T", animate=True, interpolate_multidim=True)
Draw(G_hist, mesh, "T", animate=True, interpolate_multidim=True)

WebGLScene

t =  0.10 h | Newton: 3 it | q_rad = 15.490 W/m2 | q_cond = 0.249 W/m2
t =  0.20 h | Newton: 3 it | q_rad = 15.493 W/m2 | q_cond = 0.275 W/m2
t =  0.30 h | Newton: 2 it | q_rad = 15.495 W/m2 | q_cond = 0.290 W/m2
t =  0.40 h | Newton: 2 it | q_rad = 15.497 W/m2 | q_cond = 0.299 W/m2
t =  0.50 h | Newton: 2 it | q_rad = 15.499 W/m2 | q_cond = 0.305 W/m2
t =  0.60 h | Newton: 2 it | q_rad = 15.500 W/m2 | q_cond = 0.309 W/m2
t =  0.70 h | Newton: 2 it | q_rad = 15.501 W/m2 | q_cond = 0.312 W/m2
t =  0.80 h | Newton: 2 it | q_rad = 15.502 W/m2 | q_cond = 0.313 W/m2
t =  0.90 h | Newton: 2 it | q_rad = 15.502 W/m2 | q_cond = 0.315 W/m2
t =  1.00 h | Newton: 2 it | q_rad = 15.503 W/m2 | q_cond = 0.316 W/m2
t =  1.10 h | Newton: 2 it | q_rad = 15.503 W/m2 | q_cond = 0.316 W/m2
t =  1.20 h | Newton: 2 it | q_rad = 15.504 W/m2 | q_cond = 0.317 W/m2
t =  1.30 h | Newton: 2 it | q_rad = 15.504 W/m2 | q_cond = 0.317 W/m2
t =  1.40 h | Newton: 2 it | q_rad = 15.504 W/m2 | q_cond = 0.317 W/m2
t =  1